#1. Read the Silver transaction tables + Gold dimensions

In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

orders_df = spark.table(
    f"{catalog}.ecommerce_silver.orders"
)

order_items_df = spark.table(
    f"{catalog}.ecommerce_silver.order_items"
)

payments_df = spark.table(
    f"{catalog}.ecommerce_silver.payments"
)

dim_customer = spark.table(
    f"{catalog}.ecommerce_gold.dim_customer"
)

dim_product = spark.table(
    f"{catalog}.ecommerce_gold.dim_product"
)

dim_seller = spark.table(
    f"{catalog}.ecommerce_gold.dim_seller"
)

#2. Build the fact

###Step 1 — Aggregate payments

In [0]:
payment_summary = (
    payments_df
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.first("payment_type", ignorenulls=True).alias("payment_type"),
        F.max("payment_installments").alias("payment_installments")
    )
)

###Step 2 — Build the fact using payment_summary instead of payments_df

In [0]:
fact_sales = (
    order_items_df
    .join(
        orders_df,
        order_items_df["order_id"] == orders_df["order_id"],
        "left"
    )
    .join(
        payment_summary,
        order_items_df["order_id"] == payment_summary["order_id"],
        "left"
    )
    .join(
        dim_customer,
        orders_df["customer_id"] == dim_customer["customer_id"],
        "left"
    )
    .join(
        dim_product,
        order_items_df["product_id"] == dim_product["product_id"],
        "left"
    )
    .join(
        dim_seller,
        order_items_df["seller_id"] == dim_seller["seller_id"],
        "left"
    )
)

#3. Select the final fact columns

In [0]:
gold_fact_sales = fact_sales.select(
    order_items_df["order_id"],
    order_items_df["order_item_id"],

    # Dimension foreign keys
    dim_customer["customer_key"],
    dim_product["product_key"],
    dim_seller["seller_key"],

    # Order information
    orders_df["order_status"],
    orders_df["purchase_timestamp"],
    orders_df["approved_timestamp"],
    orders_df["delivered_carrier_timestamp"],
    orders_df["delivered_customer_timestamp"],
    orders_df["estimated_delivery_timestamp"],

    # Sales measures
    order_items_df["price"],
    order_items_df["freight_value"],

    # Payment attributes
    payment_summary["payment_type"],
    payment_summary["payment_installments"]
)

#4. Write the Gold fact to ADLS

In [0]:
gold_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/gold/fact_sales"

gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

#5. Register it in Unity Catalog

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_gold.fact_sales
USING DELTA
LOCATION '{gold_path}'
""")

#6. Verification


In [0]:
spark.table(
    f"{catalog}.ecommerce_gold.fact_sales"
).show(10, truncate=False)

In [0]:
spark.table(
    f"{catalog}.ecommerce_gold.fact_sales"
).printSchema()